# Análise exploratória do corpus de e-commerce

Notebook de apoio ao benchmark de modelos clássicos de Recuperação da Informação.

O objetivo aqui é **entender o corpus antes de julgar os algoritmos**: se a
distribuição de termos, o comprimento dos documentos ou a ambiguidade do
vocabulário não forem conhecidos, qualquer diferença de desempenho entre os
modelos fica sem explicação.

Roteiro:

1. Carregar o catálogo e conferir o schema
2. Distribuição de categorias, preços e notas
3. Comprimento das descrições
4. Vocabulário e lei de Zipf
5. Polissemia e sinonímia plantadas
6. Dimensionalidade do espaço latente (LSA)

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Permite importar o projeto a partir de notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import config
from src import utils
from src.corpus_generator import corpus_statistics, load_corpus, load_qrels
from src.indexer import build_inverted_index

config.setup_logging()

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#dcdcd8"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]


def style(ax, title="", xlabel="", ylabel=""):
    """Aplica o estilo comum aos gráficos deste notebook."""
    ax.set_facecolor(SURFACE)
    ax.set_title(title, fontsize=12, color=INK, loc="left", pad=10)
    ax.set_xlabel(xlabel, fontsize=10, color=MUTED)
    ax.set_ylabel(ylabel, fontsize=10, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.8, zorder=0)
    ax.tick_params(colors=MUTED, labelsize=9)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    return ax

## 1. Carregamento e schema

Se `data/products.json` ainda não existir, `load_corpus()` gera o catálogo
na hora com a semente de `config.RANDOM_SEED`.

In [ ]:
corpus = load_corpus()
qrels = load_qrels()
frame = pd.DataFrame(corpus)

print(f"{len(frame):,} produtos · colunas: {list(frame.columns)}")
frame.head(3)

In [ ]:
stats = corpus_statistics(corpus)
for key, value in stats.items():
    if key != "categorias":
        print(f"{key:>16}: {value}")

frame[["price", "rating"]].describe().round(2)

## 2. Categorias, preços e notas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), facecolor=SURFACE)

counts = frame["category"].value_counts().sort_values()
axes[0].barh(counts.index, counts.values, color=SERIES[0], zorder=3, height=0.7)
style(axes[0], "Produtos por categoria", "produtos", "")
for y, value in enumerate(counts.values):
    axes[0].text(value + 12, y, str(value), va="center", fontsize=8.5, color=MUTED)

axes[1].hist(frame["price"], bins=60, color=SERIES[1], zorder=3)
axes[1].set_xscale("log")
style(axes[1], "Distribuição de preços (log)", "preço (R$)", "produtos")

axes[2].hist(frame["rating"], bins=np.arange(1.0, 5.2, 0.2), color=SERIES[2], zorder=3)
style(axes[2], "Distribuição de notas", "rating", "produtos")

fig.tight_layout()

## 3. Comprimento das descrições

A variação de comprimento é deliberada: é ela que dá trabalho ao parâmetro
`b` do BM25, responsável por normalizar o score pelo tamanho do documento.

In [ ]:
frame["n_palavras"] = frame["description"].str.split().str.len()
frame["n_caracteres"] = frame["description"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), facecolor=SURFACE)

axes[0].hist(frame["n_palavras"], bins=40, color=SERIES[0], zorder=3)
axes[0].axvline(frame["n_palavras"].mean(), color=SERIES[1], linewidth=2, zorder=4)
style(axes[0], "Palavras por descrição", "palavras", "produtos")
axes[0].annotate(
    f"média {frame['n_palavras'].mean():.0f}",
    xy=(frame["n_palavras"].mean(), axes[0].get_ylim()[1] * 0.9),
    xytext=(8, 0), textcoords="offset points", color=MUTED, fontsize=9,
)

order = frame.groupby("category")["n_palavras"].median().sort_values().index
axes[1].boxplot(
    [frame.loc[frame["category"] == c, "n_palavras"] for c in order],
    tick_labels=order, vert=False, patch_artist=True,
    boxprops=dict(facecolor=SERIES[2], edgecolor=MUTED, linewidth=0.8),
    medianprops=dict(color=INK, linewidth=1.5),
    flierprops=dict(markersize=2, markerfacecolor=MUTED, markeredgecolor="none"),
)
style(axes[1], "Comprimento por categoria", "palavras", "")

fig.tight_layout()
frame["n_palavras"].describe().round(1)

## 4. Vocabulário e lei de Zipf

Construímos o índice invertido — a mesma estrutura usada pelo BM25 — e
verificamos se a distribuição de frequências segue Zipf. Uma reta em escala
log-log indica que poucos termos concentram a maior parte das ocorrências,
que é exatamente a razão de existir o IDF.

In [ ]:
texts = utils.build_corpus_texts(corpus)
utils.warm_up(texts[:50])
index = build_inverted_index(corpus, texts)

collection_freq = Counter()
for term, postings in index.postings.items():
    collection_freq[term] = sum(tf for _, tf in postings)

print(f"vocabulário (após stemming): {index.vocabulary_size:,} termos")
print(f"comprimento médio indexado : {index.avg_doc_length:.1f} tokens")
print(f"tempo de indexação         : {index.indexing_time_ms:.0f} ms")

pd.DataFrame(
    [
        {"termo": t, "freq_coleção": collection_freq[t], "df": index.doc_freq[t],
         "df_%": round(100 * index.doc_freq[t] / index.n_docs, 1)}
        for t, _ in collection_freq.most_common(15)
    ]
)

In [ ]:
freqs = np.array([f for _, f in collection_freq.most_common()])
ranks = np.arange(1, len(freqs) + 1)

fig, ax = plt.subplots(figsize=(7.5, 4.6), facecolor=SURFACE)
ax.plot(ranks, freqs, color=SERIES[0], linewidth=2, zorder=3, label="observado")
ax.plot(ranks, freqs[0] / ranks, color=MUTED, linewidth=1, linestyle=(0, (4, 4)),
        zorder=2, label="Zipf ideal (1/rank)")
ax.set_xscale("log"); ax.set_yscale("log")
style(ax, "Distribuição de frequências (lei de Zipf)", "rank do termo", "frequência")
ax.legend(frameon=False, fontsize=9, labelcolor=MUTED)
fig.tight_layout()

## 5. Polissemia e sinonímia

O corpus foi construído com ambiguidades propositais. Aqui confirmamos que
elas existem de fato — é o que torna as consultas Q1 e Q5 informativas.

In [ ]:
def categorias_do_termo(termo: str) -> pd.Series:
    """Distribuição de categorias dos documentos que contêm o termo."""
    stem = utils.preprocess(termo)[0]
    docs = [doc for doc, _ in index.postings.get(stem, [])]
    return pd.Series([corpus[d]["category"] for d in docs]).value_counts()


for termo in ["mouse", "teclado", "bateria", "vela", "manga", "tablete"]:
    dist = categorias_do_termo(termo)
    resumo = ", ".join(f"{c} ({n})" for c, n in dist.head(4).items())
    print(f"{termo:>9}: {dist.sum():4d} docs em {len(dist)} categorias → {resumo}")

In [ ]:
# Sinonímia: com que frequência cada forma aparece?
pares = [("notebook", "laptop"), ("sem fio", "wireless"), ("periférico", "acessório")]
linhas = []
for a, b in pares:
    stem_a, stem_b = utils.preprocess(a)[-1], utils.preprocess(b)[-1]
    docs_a = {d for d, _ in index.postings.get(stem_a, [])}
    docs_b = {d for d, _ in index.postings.get(stem_b, [])}
    linhas.append({
        "forma A": a, "docs A": len(docs_a),
        "forma B": b, "docs B": len(docs_b),
        "só A": len(docs_a - docs_b), "só B": len(docs_b - docs_a),
        "ambas": len(docs_a & docs_b),
    })

print("Documentos que usam só uma das formas são invisíveis para a busca literal")
print("pela outra — é esse vão que o LSA tenta cobrir.")
pd.DataFrame(linhas)

In [ ]:
# Tamanho do conjunto relevante de cada consulta qualitativa
pd.DataFrame([
    {
        "query_id": spec["id"],
        "consulta": spec["query"],
        "tipo": spec["tipo"],
        "relevantes": len(qrels.get(spec["id"], {})),
        "% do corpus": round(100 * len(qrels.get(spec["id"], {})) / len(corpus), 1),
    }
    for spec in config.QUALITATIVE_QUERIES
])

## 6. Quantas dimensões o LSA precisa?

A curva de variância explicada acumulada indica quanto do sinal sobrevive à
redução de dimensionalidade e ajuda a justificar o `k = 100` adotado.

In [ ]:
from src.retrieval.lsa_search import build_lsa_index
from src.retrieval.vector_search import build_vector_index

vector_index = build_vector_index(corpus, texts)
lsa_index = build_lsa_index(corpus, texts, k=200, vector_index=vector_index)

acumulada = np.cumsum(lsa_index.svd_model.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(7.5, 4.6), facecolor=SURFACE)
ax.plot(np.arange(1, len(acumulada) + 1), acumulada * 100, color=SERIES[4], linewidth=2, zorder=3)
ax.axvline(config.LSA_COMPONENTS, color=MUTED, linewidth=1, linestyle=(0, (4, 4)), zorder=2)
style(ax, "Variância explicada acumulada pelo SVD", "dimensões latentes (k)", "% da variância")
ax.annotate(
    f"k = {config.LSA_COMPONENTS} → {acumulada[config.LSA_COMPONENTS - 1] * 100:.1f}%",
    xy=(config.LSA_COMPONENTS, acumulada[config.LSA_COMPONENTS - 1] * 100),
    xytext=(10, -18), textcoords="offset points", color=MUTED, fontsize=9.5,
)
fig.tight_layout()

print(f"vocabulário TF-IDF: {len(vector_index.vocabulary):,} termos")
print(f"k=100 retém {acumulada[99] * 100:.1f}% · k=200 retém {acumulada[-1] * 100:.1f}%")

---

## Conclusões da exploração

- O corpus tem **5.000 produtos** distribuídos em 10 categorias, com
  Informática concentrando a maior fatia (as consultas de teste vivem ali).
- As descrições variam de **200 a 500 palavras**, faixa suficiente para que a
  normalização por comprimento do BM25 tenha efeito mensurável.
- A distribuição de frequências segue Zipf, o que confirma a utilidade do IDF.
- A polissemia está presente: `mouse`, `teclado`, `bateria`, `vela` e `manga`
  aparecem em categorias semanticamente incompatíveis.
- Há documentos que usam apenas uma das formas de cada par de sinônimos —
  exatamente o vão que separa a busca literal da busca semântica.

Próximos passos: `benchmark/benchmark_scalability.py` e
`benchmark/benchmark_relevance.py`; os resultados consolidados estão em
`results/REPORT.md`.